In [1]:
import pandas as pd

from langchain import PromptTemplate
from langchain.chains import LLMChain
from langchain.chat_models import ChatOpenAI

from dotenv import load_dotenv
load_dotenv()

llm = ChatOpenAI(temperature=0, model_name='gpt-3.5-turbo-16k', request_timeout=120) 

In [5]:
df = pd.read_csv("../data/claims_justification_labels1.csv")

In [6]:
df.head()

,video_id,title,author,description,org_transcript,eng_transcript,image_base64,url,Summary_Claims,Justification,labels
0,FyQiu-o3VaI,IGL share price drop by 10.8%: Know the Reason...,5paisa,Indraprastha Gas share price (IGL) fell 10.84%...,"Hi Guys, IGL की Stock में आज 10% तक का बड़ा फॉ...","""Hi Guys, we have seen a big fall of up to 10%...",/9j/4AAQSkZJRgABAQAAAQABAAD/2wCEAAgGBggIDQgIDQ...,https://www.youtube.com/watch?v=FyQiu-o3VaI,The financial influencer claims that the IGL s...,The claim is plausible as the adoption of EVs ...,true
1,TK74cx0p-NM,Nestle India share price jumps by over 1.5% af...,5paisa,Nestle India share price jumped more than 1.5%...,[Music] hi everyone 1.5 Financial results Mar...,[Music] Hi everyone! The financial results for...,/9j/4AAQSkZJRgABAQAAAQABAAD/2wCEAAgGBgcICAgIBw...,https://www.youtube.com/watch?v=TK74cx0p-NM,The financial influencer claims that the finan...,The claim is true if the financial results for...,true
2,70ifhSt5N_c,Wipro Q2 Results Highlights #q2results #shorts,5paisa,Discover the key highlights from Wipro's Q2 FY...,"Hi guys, Vipro ke numbers ek baar phir umeet s...","""Hi guys, Vipro's numbers have once again fall...",/9j/4AAQSkZJRgABAQAAAQABAAD/2wCEAAgGBggICAgICA...,https://www.youtube.com/watch?v=70ifhSt5N_c,The financial influencer claims that Vipro's r...,The claims made by the influencer are based on...,neutral
3,zb6dOEuW6qg,Nifty/BankNifty Prediction For Tomorrow for 19...,5paisa,Here you can find the nifty predictions for 19...,पिछले कुछ दिनों में मार्केट्स में ग्रैजूल रिका...,"In the past few days, the markets witnessed gr...",/9j/4AAQSkZJRgABAQAAAQABAAD/2wCEAAgGBgkICAgICQ...,https://www.youtube.com/watch?v=zb6dOEuW6qg,The financial influencer claims that the marke...,The claims made by the influencer are based on...,neutral
4,w-neJT7eaGU,Zomato Share Price Surges to 52-Week High: Kno...,5paisa,"On 18-Oct-2023, Zomato share price reached a 5...","Hi everyone, aaj Zomato ke shares ne a 52 week...","""Hi everyone, today Zomato's shares have hit a...",/9j/4AAQSkZJRgABAQAAAQABAAD/2wCEAAgGBgcICAgIBw...,https://www.youtube.com/watch?v=w-neJT7eaGU,The financial influencer claims that Zomato's ...,The claim seems plausible as strategic partner...,true


In [7]:
df.labels.value_counts()

labels
true       62
neutral    41
false       6
Name: count, dtype: int64

In [5]:
def assign_gt_labels(sample):
   summary_claims = sample["Summary_Claims"]
   justification = sample["Justification"]

   template="""
      You are a Financial Analyst. You are given with Claims made by Financial Infuencer. \
      Your task is to classify if the Claims are true, false or neutral based on provided \
      Justification. If there are no Claims then assign neutral. \
      Restrict the response to one of the label categories. No need of any explanation.

      Claim Summary: {summary_claims}
      Justifications: {justification}

      Category: 
      """
   prompt = PromptTemplate(
      input_variables=['summary_claims', 'justification'],
      template=template
   )
   chain = LLMChain(llm=llm, prompt=prompt)
   labels = chain.run({'summary_claims':summary_claims, 'justification':justification})
   # formatted_output = parser.parse(labels)

   return labels

In [8]:
df["labels"] = df.apply(assign_gt_labels, axis=1)

In [9]:
df.head()

,video_id,title,author,description,org_transcript,eng_transcript,image_base64,url,Summary_Claims,Justification,labels
0,FyQiu-o3VaI,IGL share price drop by 10.8%: Know the Reason...,5paisa,Indraprastha Gas share price (IGL) fell 10.84%...,"Hi Guys, IGL की Stock में आज 10% तक का बड़ा फॉ...","""Hi Guys, we have seen a big fall of up to 10%...",/9j/4AAQSkZJRgABAQAAAQABAAD/2wCEAAgGBggIDQgIDQ...,https://www.youtube.com/watch?v=FyQiu-o3VaI,The financial influencer claims that the IGL s...,The claim is plausible as the adoption of EVs ...,True
1,TK74cx0p-NM,Nestle India share price jumps by over 1.5% af...,5paisa,Nestle India share price jumped more than 1.5%...,[Music] hi everyone 1.5 Financial results Mar...,[Music] Hi everyone! The financial results for...,/9j/4AAQSkZJRgABAQAAAQABAAD/2wCEAAgGBgcICAgIBw...,https://www.youtube.com/watch?v=TK74cx0p-NM,The financial influencer claims that the finan...,The claim is true if the financial results for...,True
2,70ifhSt5N_c,Wipro Q2 Results Highlights #q2results #shorts,5paisa,Discover the key highlights from Wipro's Q2 FY...,"Hi guys, Vipro ke numbers ek baar phir umeet s...","""Hi guys, Vipro's numbers have once again fall...",/9j/4AAQSkZJRgABAQAAAQABAAD/2wCEAAgGBggICAgICA...,https://www.youtube.com/watch?v=70ifhSt5N_c,The financial influencer claims that Vipro's r...,The claims made by the influencer are based on...,Neutral
3,zb6dOEuW6qg,Nifty/BankNifty Prediction For Tomorrow for 19...,5paisa,Here you can find the nifty predictions for 19...,पिछले कुछ दिनों में मार्केट्स में ग्रैजूल रिका...,"In the past few days, the markets witnessed gr...",/9j/4AAQSkZJRgABAQAAAQABAAD/2wCEAAgGBgkICAgICQ...,https://www.youtube.com/watch?v=zb6dOEuW6qg,The financial influencer claims that the marke...,The claims made by the influencer are based on...,Neutral
4,w-neJT7eaGU,Zomato Share Price Surges to 52-Week High: Kno...,5paisa,"On 18-Oct-2023, Zomato share price reached a 5...","Hi everyone, aaj Zomato ke shares ne a 52 week...","""Hi everyone, today Zomato's shares have hit a...",/9j/4AAQSkZJRgABAQAAAQABAAD/2wCEAAgGBgcICAgIBw...,https://www.youtube.com/watch?v=w-neJT7eaGU,The financial influencer claims that Zomato's ...,The claim seems plausible as strategic partner...,True


In [14]:
# df.to_csv("../data/claims_justification_labels1.csv", index=False)